# Pandas Part 3: Aggregation, GroupBy, Merging, and Reshaping in Pandas

**Focus:** Turning cleaned data into business analysis using aggregation, grouped summaries, joins, and reshaping

---

## Why this session matters

In real analytics work, once the data is reasonably clean, the next question is:

> How do we turn rows of transactions into meaningful business insight?

This is where Pandas becomes powerful.

In this session, students will learn how to:
- summarize large datasets
- answer management questions using grouped analysis
- combine multiple related tables
- reshape data for dashboards, reporting, and modeling

This is one of the most practical parts of Pandas because it mirrors real analyst work:
- sales by country
- performance by channel
- customer value by segment
- product trends by month
- merging transactions with customer and marketing data
- creating pivot-style summaries for decision-makers

---

## Business scenario

You work on the analytics team of a specialty beverage retailer.

Management wants answers such as:
- Which countries, channels, and categories drive the most revenue?
- Which customer segments are most valuable?
- Which products perform best after accounting for volume and returns?
- How does monthly revenue compare across countries?
- How much marketing spend is associated with sales by channel and country?
- What tables should be produced for executive dashboards?

To answer these questions, you will use four related datasets:

1. **transactions** — transactional sales records  
2. **customers** — customer attributes and acquisition information  
3. **products** — product attributes and supplier details  
4. **marketing_spend** — monthly marketing spend by country and channel

---

## Learning objectives

By the end of this session, students should be able to:

- Use `groupby()` to summarize data by one or more dimensions
- Apply multiple aggregation functions with `agg()`
- Compare counts, sums, averages, and rates appropriately
- Merge fact and dimension tables using `merge()`
- Diagnose merge issues such as unmatched keys
- Create pivot tables for management reporting
- Reshape wide and long tables using `melt()` and pivot operations
- Build analysis-ready summary tables for visualization and machine learning



## Imports and display settings

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Pandas version:", pd.__version__)

Pandas version: 3.0.0


## Load the datasets

This session uses four related CSV files:

- `transactions.csv`
- `customers.csv`
- `products.csv`
- `marketing_spend.csv`

This structure is intentional.  
In real organizations, the analyst rarely works with just one table.


In [5]:
transactions = pd.read_csv("transactions.csv", parse_dates=["order_date"])
customers = pd.read_csv("customers.csv", parse_dates=["signup_date"])
products = pd.read_csv("products.csv")
marketing = pd.read_csv("marketing_spend.csv")

print("transactions:", transactions.shape)
print("customers   :", customers.shape)
print("products    :", products.shape)
print("marketing   :", marketing.shape)

transactions: (2401, 16)
customers   : (649, 10)
products    : (20, 7)
marketing   : (108, 5)


In [6]:
transactions.head()

,order_id,order_date,order_month,customer_id,product_id,product_name,category,country,channel,segment,quantity,unit_price,discount,gross_sales,is_return,net_sales
0,O100000,2024-01-21,2024-01,C10214,P1010,French Press,Equipment,Australia,Marketplace,Consumer,1,24.65,0.00,24.65,0,24.65
1,O100001,2024-01-01,2024-01,C10401,P1008,Espresso Blend,Coffee,Australia,Marketplace,Consumer,2,16.77,0.00,33.54,0,33.54
2,O100002,2024-01-16,2024-01,C10147,P1004,Colombian Beans,Coffee,Australia,Marketplace,Consumer,2,14.96,0.00,29.92,0,29.92
3,O100003,2024-01-21,2024-01,C10077,P1002,Ceramic Mug,Merch,Australia,Marketplace,Consumer,2,12.65,0.00,25.30,0,25.30
4,O100004,2024-01-15,2024-01,C10279,P1019,Travel Mug,Merch,Australia,Marketplace,Consumer,1,18.87,0.00,18.87,0,18.87


In [7]:
customers.head()

,customer_id,home_country,home_city,dominant_segment,total_orders,lifetime_revenue,signup_date,customer_tier,loyalty_member,acquisition_channel
0,C10000,Germany,Austin,Consumer,6,627.09,2023-07-13,Gold,Yes,Organic Search
1,C10001,United Kingdom,Birmingham,Consumer,3,155.50,2022-01-08,Gold,Yes,Referral
2,C10002,United States,San Jose,Consumer,10,426.45,2022-05-12,Gold,No,Organic Search
3,C10003,United States,Vancouver,Small Business,5,264.68,2022-12-07,Gold,Yes,Email
4,C10004,Australia,Chicago,Consumer,4,231.00,2022-09-09,Bronze,Yes,Social


In [8]:
products.head()

,sku_name,category,typical_unit_price,product_id,supplier,launch_year,margin_band
0,Biscotti Jar,Bakery,11.50,P1000,Maple Bridge,2022,High
1,Burr Grinder,Equipment,79.00,P1001,BlueRiver Imports,2023,Medium
2,Ceramic Mug,Merch,14.00,P1002,Summit Trading,2023,Medium
3,Cold Brew Pack,Coffee,14.00,P1003,NorthPeak Supply,2024,Medium
4,Colombian Beans,Coffee,16.50,P1004,Summit Trading,2023,High


In [9]:
marketing.head()

,order_month,country,channel,marketing_spend,campaign_count
0,2024-01,Australia,Marketplace,"1,585.96",1
1,2024-01,Australia,Online,"1,946.98",3
2,2024-01,Australia,Store Pickup,615.43,5
3,2024-01,Canada,Marketplace,"1,073.16",5
4,2024-01,Canada,Online,"1,689.49",2


## First question: what does each row represent?

A strong analyst always starts by clarifying granularity.

### Granularity in these tables
- **transactions:** one row per transaction record
- **customers:** one row per customer
- **products:** one row per product
- **marketing:** one row per month-country-channel combination

### Discussion prompt
Why does granularity matter for merging and aggregation?


In [10]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 2401 entries, 0 to 2400
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      2401 non-null   str           
 1   order_date    2401 non-null   datetime64[us]
 2   order_month   2401 non-null   str           
 3   customer_id   2401 non-null   str           
 4   product_id    2401 non-null   str           
 5   product_name  2401 non-null   str           
 6   category      2401 non-null   str           
 7   country       2401 non-null   str           
 8   channel       2401 non-null   str           
 9   segment       2401 non-null   str           
 10  quantity      2401 non-null   int64         
 11  unit_price    2401 non-null   float64       
 12  discount      2401 non-null   float64       
 13  gross_sales   2401 non-null   float64       
 14  is_return     2401 non-null   int64         
 15  net_sales     2401 non-null   float64       
dtyp

In [11]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 649 entries, 0 to 648
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          649 non-null    str           
 1   home_country         649 non-null    str           
 2   home_city            649 non-null    str           
 3   dominant_segment     649 non-null    str           
 4   total_orders         649 non-null    int64         
 5   lifetime_revenue     649 non-null    float64       
 6   signup_date          649 non-null    datetime64[us]
 7   customer_tier        649 non-null    str           
 8   loyalty_member       649 non-null    str           
 9   acquisition_channel  649 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(1), str(7)
memory usage: 50.8 KB


In [12]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   sku_name            20 non-null     str    
 1   category            20 non-null     str    
 2   typical_unit_price  20 non-null     float64
 3   product_id          20 non-null     str    
 4   supplier            20 non-null     str    
 5   launch_year         20 non-null     int64  
 6   margin_band         20 non-null     str    
dtypes: float64(1), int64(1), str(5)
memory usage: 1.2 KB


In [13]:
marketing.info()

<class 'pandas.DataFrame'>
RangeIndex: 108 entries, 0 to 107
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_month      108 non-null    str    
 1   country          108 non-null    str    
 2   channel          108 non-null    str    
 3   marketing_spend  108 non-null    float64
 4   campaign_count   108 non-null    int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 4.3 KB


## GroupBy basics: from rows to summaries

This is the heart of practical Pandas work.

The pattern is:

```python
df.groupby("some_column")["numeric_column"].aggregation()
```

Examples:
- revenue by country
- revenue by category
- transaction count by channel


In [14]:
transactions.groupby("country")["net_sales"].sum().sort_values(ascending=False)

country
United States    23,466.65
United Kingdom   13,226.06
Germany          10,761.01
Canada           10,360.75
Australia         9,449.97
France            7,863.45
Name: net_sales, dtype: float64

In [15]:
transactions.groupby("category")["net_sales"].sum().sort_values(ascending=False)

category
Coffee      17,056.21
Equipment   15,903.14
Bakery      15,176.26
Merch       14,186.71
Tea         12,805.57
Name: net_sales, dtype: float64

In [16]:
transactions.groupby("channel")["order_id"].count().sort_values(ascending=False)

channel
Online          1149
Marketplace      786
Store Pickup     466
Name: order_id, dtype: int64

### Discussion prompt

These three outputs answer different business questions:
- total sales
- product mix
- transaction volume

Students should get used to matching the metric to the question.


## More useful grouped metrics with `agg()`

Most real analysis needs more than one metric.

For example, management often wants:
- number of transactions
- total sales
- average sales
- median sales


In [17]:
transactions.groupby("country")["net_sales"].agg(["count", "sum", "mean", "median"]).sort_values("sum", ascending=False)

,count,sum,mean,median
country,,,,
United States,601,"23,466.65",39.05,34.77
United Kingdom,387,"13,226.06",34.18,32.85
Germany,355,"10,761.01",30.31,27.08
Canada,383,"10,360.75",27.05,25.41
Australia,358,"9,449.97",26.40,23.67
France,317,"7,863.45",24.81,22.70


### Named aggregation syntax

This style is especially useful because it produces clearer column names.


In [18]:
country_summary = (
    transactions.groupby("country")
    .agg(
        transactions=("order_id", "count"),
        customers=("customer_id", "nunique"),
        total_sales=("net_sales", "sum"),
        avg_sales=("net_sales", "mean"),
        median_sales=("net_sales", "median"),
        return_rows=("is_return", "sum")
    )
    .sort_values("total_sales", ascending=False)
)
country_summary

,transactions,customers,total_sales,avg_sales,median_sales,return_rows
country,,,,,,
United States,601,250,"23,466.65",39.05,34.77,21
United Kingdom,387,99,"13,226.06",34.18,32.85,23
Germany,355,54,"10,761.01",30.31,27.08,17
Canada,383,81,"10,360.75",27.05,25.41,28
Australia,358,54,"9,449.97",26.40,23.67,20
France,317,38,"7,863.45",24.81,22.70,16


## Multi-level grouping

Real business questions usually involve more than one dimension:
- sales by country and channel
- sales by country and month
- sales by segment and category


In [19]:
transactions.groupby(["country", "channel"])["net_sales"].sum().sort_values(ascending=False).head(20)

country         channel     
United States   Online         11,975.11
United Kingdom  Online          6,728.07
United States   Marketplace     6,645.01
Germany         Online          5,547.82
Canada          Online          5,305.11
United States   Store Pickup    4,846.53
Australia       Online          4,506.46
France          Online          4,151.97
United Kingdom  Marketplace     3,850.67
Canada          Marketplace     3,648.93
Australia       Marketplace     3,136.10
Germany         Marketplace     2,816.04
United Kingdom  Store Pickup    2,647.32
Germany         Store Pickup    2,397.15
France          Marketplace     2,142.46
Australia       Store Pickup    1,807.41
France          Store Pickup    1,569.02
Canada          Store Pickup    1,406.71
Name: net_sales, dtype: float64

In [20]:
transactions.groupby(["order_month", "country"])["net_sales"].sum().head(15)

order_month  country       
2024-01      Australia        1,396.81
             Canada           1,366.57
             France           1,104.51
             Germany          2,183.68
             United Kingdom   2,573.29
             United States    3,640.03
2024-02      Australia        1,655.06
             Canada           1,742.50
             France           1,192.81
             Germany          1,615.85
             United Kingdom   2,390.16
             United States    4,221.71
2024-03      Australia        1,797.76
             Canada           1,831.11
             France           1,482.02
Name: net_sales, dtype: float64

In [21]:
transactions.groupby(["segment", "category"])["net_sales"].sum().sort_values(ascending=False)

segment         category 
Consumer        Coffee      15,664.00
                Bakery      14,221.04
                Equipment   13,768.62
                Merch       13,358.89
                Tea         11,342.89
Small Business  Equipment    2,103.32
                Tea          1,381.97
                Coffee       1,247.19
                Bakery         832.91
                Merch          751.39
Corporate       Coffee         145.02
                Bakery         122.31
                Tea             80.71
                Merch           76.43
                Equipment       31.20
Name: net_sales, dtype: float64

### Questions
- Which grouped table is easiest to interpret?
- When does a MultiIndex become harder to read?
- When might resetting the index be useful?


In [ ]:
summary_reset = (
    transactions.groupby(["country", "channel"])
    .agg(total_sales=("net_sales", "sum"), transactions=("order_id", "count"))
    .reset_index() # convert from multi-index to regular index
    .sort_values(["country", "total_sales"], ascending=[True, False])
)
summary_reset.head(15)

,country,channel,total_sales,transactions
1,Australia,Online,"4,506.46",167
0,Australia,Marketplace,"3,136.10",126
2,Australia,Store Pickup,"1,807.41",65
4,Canada,Online,"5,305.11",187
3,Canada,Marketplace,"3,648.93",131
5,Canada,Store Pickup,"1,406.71",65
7,France,Online,"4,151.97",154
6,France,Marketplace,"2,142.46",100
8,France,Store Pickup,"1,569.02",63
10,Germany,Online,"5,547.82",178


## Grouped analysis for real questions

- Which country-channel combinations generate the most total sales?
- Which categories have the highest average transaction value?
- Which months look strongest in each country?


In [23]:
transactions.groupby(["country", "channel"])["net_sales"].sum().sort_values(ascending=False).head(15)

country         channel     
United States   Online         11,975.11
United Kingdom  Online          6,728.07
United States   Marketplace     6,645.01
Germany         Online          5,547.82
Canada          Online          5,305.11
United States   Store Pickup    4,846.53
Australia       Online          4,506.46
France          Online          4,151.97
United Kingdom  Marketplace     3,850.67
Canada          Marketplace     3,648.93
Australia       Marketplace     3,136.10
Germany         Marketplace     2,816.04
United Kingdom  Store Pickup    2,647.32
Germany         Store Pickup    2,397.15
France          Marketplace     2,142.46
Name: net_sales, dtype: float64

In [24]:
transactions.groupby("category")["net_sales"].mean().sort_values(ascending=False)

category
Equipment   41.63
Bakery      33.35
Coffee      29.41
Merch       29.01
Tea         25.87
Name: net_sales, dtype: float64

In [25]:
monthly_country = (
    transactions.groupby(["order_month", "country"])
    .agg(total_sales=("net_sales", "sum"))
    .reset_index()
    .sort_values(["country", "order_month"])
)
monthly_country.head(20)

,order_month,country,total_sales
0,2024-01,Australia,"1,396.81"
6,2024-02,Australia,"1,655.06"
12,2024-03,Australia,"1,797.76"
18,2024-04,Australia,"1,437.84"
24,2024-05,Australia,"1,601.96"
30,2024-06,Australia,"1,560.54"
1,2024-01,Canada,"1,366.57"
7,2024-02,Canada,"1,742.50"
13,2024-03,Canada,"1,831.11"
19,2024-04,Canada,"1,857.13"


## Useful groupby patterns
### Count distinct values in groups
Example: unique customers by country


In [26]:
transactions.groupby("country")["customer_id"].nunique().sort_values(ascending=False)

country
United States     250
United Kingdom     99
Canada             81
Australia          54
Germany            54
France             38
Name: customer_id, dtype: int64

### Use boolean columns for rates and counts

Because `True` behaves like `1` and `False` like `0`, boolean columns are extremely useful.

Example: return rate by category


In [27]:
transactions.groupby("category")["is_return"].mean().sort_values(ascending=False)

category
Merch       0.08
Equipment   0.07
Coffee      0.05
Bakery      0.04
Tea         0.03
Name: is_return, dtype: float64

### Combine count, sum, and rate in one summary


In [28]:
category_summary = (
    transactions.groupby("category")
    .agg(
        transaction_rows=("order_id", "count"),
        unique_products=("product_name", "nunique"),
        total_sales=("net_sales", "sum"),
        avg_sales=("net_sales", "mean"),
        return_rate=("is_return", "mean")
    )
    .sort_values("total_sales", ascending=False)
)
category_summary

,transaction_rows,unique_products,total_sales,avg_sales,return_rate
category,,,,,
Coffee,580,4,"17,056.21",29.41,0.05
Equipment,382,4,"15,903.14",41.63,0.07
Bakery,455,4,"15,176.26",33.35,0.04
Merch,489,4,"14,186.71",29.01,0.08
Tea,495,4,"12,805.57",25.87,0.03


## Mini exercise 1 — GroupBy practice

Write code to answer:
1. Which segment has the highest total sales?
2. Which channel has the highest average sales per transaction?
3. Which country has the most unique customers?
4. Which month has the highest total sales?


In [ ]:
# 1
transactions.groupby("segment")["net_sales"].sum().sort_values(ascending=False)

# 2
transactions.groupby("channel")["net_sales"].mean().sort_values(ascending=False)

# 3
transactions.groupby("country")["customer_id"].nunique().sort_values(ascending=False)

# 4
transactions.groupby("order_month")["net_sales"].sum().sort_values(ascending=False)

## Moving beyond one table: why merging matters

In almost every real analytics project:
- the transaction table does not contain everything
- customer details live elsewhere
- product attributes live elsewhere
- marketing or operational data lives elsewhere

To answer richer questions, we need joins.


### Merge vocabulary

- **left join**: keep all rows from the left table
- **inner join**: keep only matching rows
- **right join**: keep all rows from the right table
- **outer join**: keep everything from both

For analytics, `left` and `inner` are the most common.


In [ ]:
transactions[["customer_id", "country", "channel", "net_sales"]].head()

In [ ]:
customers[["customer_id", "customer_tier", "loyalty_member", "acquisition_channel"]].head()

## Merge transactions with customers

This lets us ask better questions:
- Do Gold customers spend more?
- Which acquisition channels bring more valuable customers?
- Are loyalty members worth more?


In [31]:
# Merge transactions with customers on customer_id, keeping all transactions and adding customer info where available. 
# The _merge column indicates whether a match was found in customers for each transaction.
tx_cust = transactions.merge(customers, on="customer_id", how="left", indicator=True) 
tx_cust.head()

,order_id,order_date,order_month,customer_id,product_id,product_name,category,country,channel,segment,quantity,unit_price,discount,gross_sales,is_return,net_sales,home_country,home_city,dominant_segment,total_orders,lifetime_revenue,signup_date,customer_tier,loyalty_member,acquisition_channel,_merge
0,O100000,2024-01-21,2024-01,C10214,P1010,French Press,Equipment,Australia,Marketplace,Consumer,1,24.65,0.00,24.65,0,24.65,Australia,Berlin,Consumer,7,405.60,2022-02-19,Silver,Yes,Referral,both
1,O100001,2024-01-01,2024-01,C10401,P1008,Espresso Blend,Coffee,Australia,Marketplace,Consumer,2,16.77,0.00,33.54,0,33.54,Australia,Hamburg,Consumer,10,645.00,2023-02-12,Gold,No,Organic Search,both
2,O100002,2024-01-16,2024-01,C10147,P1004,Colombian Beans,Coffee,Australia,Marketplace,Consumer,2,14.96,0.00,29.92,0,29.92,Australia,Austin,Consumer,6,265.85,2023-08-19,Bronze,Yes,Email,both
3,O100003,2024-01-21,2024-01,C10077,P1002,Ceramic Mug,Merch,Australia,Marketplace,Consumer,2,12.65,0.00,25.30,0,25.30,Australia,Berlin,Consumer,5,147.90,2023-07-03,Bronze,Yes,Referral,both
4,O100004,2024-01-15,2024-01,C10279,P1019,Travel Mug,Merch,Australia,Marketplace,Consumer,1,18.87,0.00,18.87,0,18.87,Australia,Brisbane,Consumer,6,339.70,2022-02-25,Silver,No,Email,both


In [32]:
tx_cust["_merge"].value_counts()

_merge
both          2401
left_only        0
right_only       0
Name: count, dtype: int64

### `indicator=True`

The `indicator=True` option is excellent for debugging.

It shows whether rows matched:
- `both`
- `left_only`
- `right_only`

Students should learn to verify merges instead of assuming they worked.


In [38]:
tx_cust = tx_cust.drop(columns="_merge")
tx_cust.shape

(2401, 25)

## Business questions with merged customer data


In [33]:
tx_cust.groupby("customer_tier")["net_sales"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)

,count,sum,mean
customer_tier,,,
Bronze,1179,"37,400.82",31.72
Silver,736,"22,234.97",30.21
Gold,486,"15,492.10",31.88


In [34]:
tx_cust.groupby("loyalty_member")["net_sales"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)

,count,sum,mean
loyalty_member,,,
Yes,1336,"41,607.29",31.14
No,1065,"33,520.60",31.47


In [35]:
tx_cust.groupby("acquisition_channel")["net_sales"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)

,count,sum,mean
acquisition_channel,,,
Organic Search,666,"21,191.17",31.82
Paid Search,472,"15,319.02",32.46
Social,436,"14,455.94",33.16
Referral,449,"13,104.83",29.19
Email,378,"11,056.93",29.25


### Questions

Which metric matters most when evaluating acquisition channels?
- total sales?
- average sales?
- number of orders?
- number of unique customers?

There is no single correct answer; it depends on the business goal.


##  Merge in product attributes

Now attach product metadata such as supplier and margin band.


In [36]:
products.head()

,sku_name,category,typical_unit_price,product_id,supplier,launch_year,margin_band
0,Biscotti Jar,Bakery,11.50,P1000,Maple Bridge,2022,High
1,Burr Grinder,Equipment,79.00,P1001,BlueRiver Imports,2023,Medium
2,Ceramic Mug,Merch,14.00,P1002,Summit Trading,2023,Medium
3,Cold Brew Pack,Coffee,14.00,P1003,NorthPeak Supply,2024,Medium
4,Colombian Beans,Coffee,16.50,P1004,Summit Trading,2023,High


In [39]:
tx_full = tx_cust.merge(
    products[["product_id", "supplier", "margin_band", "launch_year"]],
    on="product_id",
    how="left",
    indicator=True
)
tx_full["_merge"].value_counts()

_merge
both          2401
left_only        0
right_only       0
Name: count, dtype: int64

In [40]:
tx_full = tx_full.drop(columns="_merge")
tx_full.head()

,order_id,order_date,order_month,customer_id,product_id,product_name,category,country,channel,segment,quantity,unit_price,discount,gross_sales,is_return,net_sales,home_country,home_city,dominant_segment,total_orders,lifetime_revenue,signup_date,customer_tier,loyalty_member,acquisition_channel,supplier,margin_band,launch_year
0,O100000,2024-01-21,2024-01,C10214,P1010,French Press,Equipment,Australia,Marketplace,Consumer,1,24.65,0.00,24.65,0,24.65,Australia,Berlin,Consumer,7,405.60,2022-02-19,Silver,Yes,Referral,Aurora Goods,Medium,2023
1,O100001,2024-01-01,2024-01,C10401,P1008,Espresso Blend,Coffee,Australia,Marketplace,Consumer,2,16.77,0.00,33.54,0,33.54,Australia,Hamburg,Consumer,10,645.00,2023-02-12,Gold,No,Organic Search,BlueRiver Imports,Low,2022
2,O100002,2024-01-16,2024-01,C10147,P1004,Colombian Beans,Coffee,Australia,Marketplace,Consumer,2,14.96,0.00,29.92,0,29.92,Australia,Austin,Consumer,6,265.85,2023-08-19,Bronze,Yes,Email,Summit Trading,High,2023
3,O100003,2024-01-21,2024-01,C10077,P1002,Ceramic Mug,Merch,Australia,Marketplace,Consumer,2,12.65,0.00,25.30,0,25.30,Australia,Berlin,Consumer,5,147.90,2023-07-03,Bronze,Yes,Referral,Summit Trading,Medium,2023
4,O100004,2024-01-15,2024-01,C10279,P1019,Travel Mug,Merch,Australia,Marketplace,Consumer,1,18.87,0.00,18.87,0,18.87,Australia,Brisbane,Consumer,6,339.70,2022-02-25,Silver,No,Email,Aurora Goods,Low,2022


## Business questions with product enrichment


In [41]:
tx_full.groupby("supplier")["net_sales"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)

,count,sum,mean
supplier,,,
BlueRiver Imports,815,"29,242.41",35.88
Summit Trading,490,"14,764.74",30.13
NorthPeak Supply,514,"13,795.78",26.84
Aurora Goods,363,"10,883.91",29.98
Maple Bridge,219,"6,441.05",29.41


In [42]:
tx_full.groupby("margin_band")["net_sales"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False)

,count,sum,mean
margin_band,,,
Medium,1140,"33,092.64",29.03
High,715,"25,051.71",35.04
Low,546,"16,983.54",31.11


In [43]:
tx_full.groupby(["category", "margin_band"])["net_sales"].sum().sort_values(ascending=False).head(20)

category   margin_band
Bakery     High          12,685.91
Equipment  Medium        12,142.10
Merch      Medium         9,666.43
Coffee     High           8,604.76
Tea        Low            7,842.33
           Medium         4,963.24
Coffee     Low            4,620.93
Merch      Low            4,520.28
Coffee     Medium         3,830.52
Equipment  High           3,761.04
Bakery     Medium         2,490.35
Name: net_sales, dtype: float64

## Merge monthly marketing spend

Marketing spend is at a different granularity:
- month
- country
- channel

That means the correct merge key is **not** just one column.


In [44]:
marketing.head()

,order_month,country,channel,marketing_spend,campaign_count
0,2024-01,Australia,Marketplace,"1,585.96",1
1,2024-01,Australia,Online,"1,946.98",3
2,2024-01,Australia,Store Pickup,615.43,5
3,2024-01,Canada,Marketplace,"1,073.16",5
4,2024-01,Canada,Online,"1,689.49",2


In [45]:
tx_all = tx_full.merge(
    marketing,
    on=["order_month", "country", "channel"],
    how="left",
    indicator=True
)
tx_all["_merge"].value_counts()

_merge
both          2401
left_only        0
right_only       0
Name: count, dtype: int64

In [46]:
tx_all = tx_all.drop(columns="_merge")
tx_all.head()

,order_id,order_date,order_month,customer_id,product_id,product_name,category,country,channel,segment,quantity,unit_price,discount,gross_sales,is_return,net_sales,home_country,home_city,dominant_segment,total_orders,lifetime_revenue,signup_date,customer_tier,loyalty_member,acquisition_channel,supplier,margin_band,launch_year,marketing_spend,campaign_count
0,O100000,2024-01-21,2024-01,C10214,P1010,French Press,Equipment,Australia,Marketplace,Consumer,1,24.65,0.00,24.65,0,24.65,Australia,Berlin,Consumer,7,405.60,2022-02-19,Silver,Yes,Referral,Aurora Goods,Medium,2023,"1,585.96",1
1,O100001,2024-01-01,2024-01,C10401,P1008,Espresso Blend,Coffee,Australia,Marketplace,Consumer,2,16.77,0.00,33.54,0,33.54,Australia,Hamburg,Consumer,10,645.00,2023-02-12,Gold,No,Organic Search,BlueRiver Imports,Low,2022,"1,585.96",1
2,O100002,2024-01-16,2024-01,C10147,P1004,Colombian Beans,Coffee,Australia,Marketplace,Consumer,2,14.96,0.00,29.92,0,29.92,Australia,Austin,Consumer,6,265.85,2023-08-19,Bronze,Yes,Email,Summit Trading,High,2023,"1,585.96",1
3,O100003,2024-01-21,2024-01,C10077,P1002,Ceramic Mug,Merch,Australia,Marketplace,Consumer,2,12.65,0.00,25.30,0,25.30,Australia,Berlin,Consumer,5,147.90,2023-07-03,Bronze,Yes,Referral,Summit Trading,Medium,2023,"1,585.96",1
4,O100004,2024-01-15,2024-01,C10279,P1019,Travel Mug,Merch,Australia,Marketplace,Consumer,1,18.87,0.00,18.87,0,18.87,Australia,Brisbane,Consumer,6,339.70,2022-02-25,Silver,No,Email,Aurora Goods,Low,2022,"1,585.96",1


### Questions

Why would it be wrong to merge marketing spend on only `country` or only `channel`?

This is one of the most important data-integration ideas in analytics:
**the merge keys must match the logic of the data grain.**


## Use merged data for richer analysis

Now we can relate sales and marketing dimensions in the same table.


In [47]:
marketing_effect = (
    tx_all.groupby(["order_month", "country", "channel"])
    .agg(
        total_sales=("net_sales", "sum"),
        transactions=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        marketing_spend=("marketing_spend", "max")
    )
    .reset_index()
)
marketing_effect.head(15)

,order_month,country,channel,total_sales,transactions,unique_customers,marketing_spend
0,2024-01,Australia,Marketplace,548.87,23,19,"1,585.96"
1,2024-01,Australia,Online,671.08,29,20,"1,946.98"
2,2024-01,Australia,Store Pickup,176.86,9,9,615.43
3,2024-01,Canada,Marketplace,556.71,16,12,"1,073.16"
4,2024-01,Canada,Online,750.58,25,25,"1,689.49"
5,2024-01,Canada,Store Pickup,59.28,12,12,813.22
6,2024-01,France,Marketplace,372.71,17,15,"1,173.69"
7,2024-01,France,Online,470.34,19,17,"1,309.59"
8,2024-01,France,Store Pickup,261.46,9,8,633.11
9,2024-01,Germany,Marketplace,717.52,20,15,"1,357.57"


In [48]:
marketing_effect["sales_to_spend_ratio"] = (marketing_effect["total_sales"] / marketing_effect["marketing_spend"]).round(2)
marketing_effect.sort_values("sales_to_spend_ratio", ascending=False).head(20)

,order_month,country,channel,total_sales,transactions,unique_customers,marketing_spend,sales_to_spend_ratio
14,2024-01,United Kingdom,Store Pickup,722.97,14,11,988.12,0.73
101,2024-06,Germany,Store Pickup,581.98,12,11,812.96,0.72
70,2024-04,United States,Online,"1,965.63",41,36,"2,769.47",0.71
106,2024-06,United States,Online,"1,616.34",36,30,"2,488.56",0.65
52,2024-03,United States,Online,"2,233.13",51,45,"3,455.12",0.65
48,2024-03,United Kingdom,Marketplace,756.16,17,15,"1,185.73",0.64
88,2024-05,United States,Online,"1,820.16",42,40,"2,861.03",0.64
34,2024-02,United States,Online,"2,388.37",55,52,"3,786.11",0.63
107,2024-06,United States,Store Pickup,845.22,20,20,"1,378.02",0.61
65,2024-04,Germany,Store Pickup,341.74,8,6,565.21,0.60


### Note

This is not true ROI analysis yet, but it is a good practical example of how joins enable richer metrics.


## Pivot tables for dashboard-style reporting

Pivot tables are extremely useful when management wants a matrix view, such as:
- months in rows, countries in columns
- categories in rows, channels in columns


In [49]:
pd.pivot_table(
    transactions,
    index="order_month",
    columns="country",
    values="net_sales",
    aggfunc="sum",
    fill_value=0
)

country,Australia,Canada,France,Germany,United Kingdom,United States
order_month,,,,,,
2024-01,"1,396.81","1,366.57","1,104.51","2,183.68","2,573.29","3,640.03"
2024-02,"1,655.06","1,742.50","1,192.81","1,615.85","2,390.16","4,221.71"
2024-03,"1,797.76","1,831.11","1,482.02","1,730.17","1,953.65","4,213.27"
2024-04,"1,437.84","1,857.13","1,196.34","1,832.66","2,354.35","4,393.60"
2024-05,"1,601.96","1,726.57","1,388.46","1,634.19","2,225.57","3,578.62"
2024-06,"1,560.54","1,836.87","1,499.31","1,764.46","1,729.04","3,419.42"


In [50]:
pd.pivot_table(
    transactions,
    index="category",
    columns="channel",
    values="net_sales",
    aggfunc=["sum", "mean"],
    fill_value=0
)

sum                               mean                    
channel   Marketplace    Online Store Pickup Marketplace Online Store Pickup
category                                                                    
Bakery       5,371.38  5,692.98     4,111.90       31.05  33.89        36.07
Coffee       3,782.91  9,685.17     3,588.13       24.56  30.84        32.04
Equipment    3,109.88 10,752.62     2,040.64       36.16  44.99        35.80
Merch        7,085.85  5,000.52     2,100.34       28.92  29.76        27.64
Tea          2,889.19  7,083.25     2,833.13       22.57  27.24        26.48

### Why pivot tables matter

Pivot tables make data easier to read for:
- managers
- dashboard builders
- presentation slides
- quick comparison across dimensions


## Reshaping: wide vs long format

Students need to understand that tables can be reshaped depending on the analysis task.

- **wide format** often works well for reporting
- **long format** often works better for plotting and modeling


In [51]:
monthly_country_wide = pd.pivot_table(
    transactions,
    index="order_month",
    columns="country",
    values="net_sales",
    aggfunc="sum",
    fill_value=0
)
monthly_country_wide.head()

country,Australia,Canada,France,Germany,United Kingdom,United States
order_month,,,,,,
2024-01,"1,396.81","1,366.57","1,104.51","2,183.68","2,573.29","3,640.03"
2024-02,"1,655.06","1,742.50","1,192.81","1,615.85","2,390.16","4,221.71"
2024-03,"1,797.76","1,831.11","1,482.02","1,730.17","1,953.65","4,213.27"
2024-04,"1,437.84","1,857.13","1,196.34","1,832.66","2,354.35","4,393.60"
2024-05,"1,601.96","1,726.57","1,388.46","1,634.19","2,225.57","3,578.62"


### Convert wide back to long with `melt()`


In [52]:
monthly_country_long = (
    monthly_country_wide
    .reset_index()
    .melt(id_vars="order_month", var_name="country", value_name="monthly_sales")
)
monthly_country_long.head(12)

,order_month,country,monthly_sales
0,2024-01,Australia,"1,396.81"
1,2024-02,Australia,"1,655.06"
2,2024-03,Australia,"1,797.76"
3,2024-04,Australia,"1,437.84"
4,2024-05,Australia,"1,601.96"
5,2024-06,Australia,"1,560.54"
6,2024-01,Canada,"1,366.57"
7,2024-02,Canada,"1,742.50"
8,2024-03,Canada,"1,831.11"
9,2024-04,Canada,"1,857.13"


### Questions

Why might long format be better for:
- line charts
- faceted plots
- grouped statistical models


## Another reshaping example: category-channel summary


In [53]:
cat_channel = pd.pivot_table(
    transactions,
    index="category",
    columns="channel",
    values="net_sales",
    aggfunc="sum",
    fill_value=0
)
cat_channel

channel,Marketplace,Online,Store Pickup
category,,,
Bakery,"5,371.38","5,692.98","4,111.90"
Coffee,"3,782.91","9,685.17","3,588.13"
Equipment,"3,109.88","10,752.62","2,040.64"
Merch,"7,085.85","5,000.52","2,100.34"
Tea,"2,889.19","7,083.25","2,833.13"


In [54]:
cat_channel_long = (
    cat_channel
    .reset_index()
    .melt(id_vars="category", var_name="channel", value_name="total_sales")
)
cat_channel_long.head(12)

,category,channel,total_sales
0,Bakery,Marketplace,"5,371.38"
1,Coffee,Marketplace,"3,782.91"
2,Equipment,Marketplace,"3,109.88"
3,Merch,Marketplace,"7,085.85"
4,Tea,Marketplace,"2,889.19"
5,Bakery,Online,"5,692.98"
6,Coffee,Online,"9,685.17"
7,Equipment,Online,"10,752.62"
8,Merch,Online,"5,000.52"
9,Tea,Online,"7,083.25"


## Multi-metric executive summary tables

A strong analytics notebook should be able to produce polished summary tables quickly.


In [55]:
executive_country = (
    tx_all.groupby("country")
    .agg(
        total_sales=("net_sales", "sum"),
        transactions=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        avg_sales=("net_sales", "mean"),
        return_rate=("is_return", "mean"),
        avg_marketing_spend=("marketing_spend", "mean")
    )
    .assign(
        sales_per_customer=lambda x: x["total_sales"] / x["unique_customers"]
    )
    .sort_values("total_sales", ascending=False)
)
executive_country

,total_sales,transactions,unique_customers,avg_sales,return_rate,avg_marketing_spend,sales_per_customer
country,,,,,,,
United States,"23,466.65",601,250,39.05,0.03,"2,559.70",93.87
United Kingdom,"13,226.06",387,99,34.18,0.06,"1,661.24",133.60
Germany,"10,761.01",355,54,30.31,0.05,"1,601.42",199.28
Canada,"10,360.75",383,81,27.05,0.07,"1,709.87",127.91
Australia,"9,449.97",358,54,26.40,0.06,"1,543.73",175.00
France,"7,863.45",317,38,24.81,0.05,"1,398.99",206.93


In [56]:
executive_segment = (
    tx_all.groupby(["segment", "customer_tier"])
    .agg(
        total_sales=("net_sales", "sum"),
        transactions=("order_id", "count"),
        avg_sales=("net_sales", "mean"),
        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
    .sort_values(["segment", "total_sales"], ascending=[True, False])
)
executive_segment.head(20)

,segment,customer_tier,total_sales,transactions,avg_sales,unique_customers
0,Consumer,Bronze,"33,843.35",1078,31.39,236
2,Consumer,Silver,"20,803.32",697,29.85,175
1,Consumer,Gold,"13,708.77",429,31.96,113
3,Corporate,Bronze,216.88,5,43.38,3
5,Corporate,Silver,128.28,6,21.38,2
4,Corporate,Gold,110.51,2,55.26,1
6,Small Business,Bronze,"3,340.59",96,34.80,21
7,Small Business,Gold,"1,672.82",55,30.41,12
8,Small Business,Silver,"1,303.37",33,39.50,13


## Mini exercise 2 — Merging and reshaping practice

Use the merged datasets to answer:

1. Which customer tier has the highest total sales?
2. Which supplier has the highest average sales per transaction?
3. Create a pivot table of total sales by month and channel.
4. Reshape that pivot table back to long format.
5. Which country-channel-month combinations have the highest sales-to-spend ratio?


In [57]:
# 1
tx_all.groupby("customer_tier")["net_sales"].sum().sort_values(ascending=False)

# 2
tx_all.groupby("supplier")["net_sales"].mean().sort_values(ascending=False)

# 3
month_channel_pivot = pd.pivot_table(
    tx_all,
    index="order_month",
    columns="channel",
    values="net_sales",
    aggfunc="sum",
    fill_value=0
)
month_channel_pivot

# 4
month_channel_long = month_channel_pivot.reset_index().melt(
    id_vars="order_month",
    var_name="channel",
    value_name="total_sales"
)
month_channel_long.head()

# 5
marketing_effect.sort_values("sales_to_spend_ratio", ascending=False).head(10)

,order_month,country,channel,total_sales,transactions,unique_customers,marketing_spend,sales_to_spend_ratio
14,2024-01,United Kingdom,Store Pickup,722.97,14,11,988.12,0.73
101,2024-06,Germany,Store Pickup,581.98,12,11,812.96,0.72
70,2024-04,United States,Online,"1,965.63",41,36,"2,769.47",0.71
106,2024-06,United States,Online,"1,616.34",36,30,"2,488.56",0.65
52,2024-03,United States,Online,"2,233.13",51,45,"3,455.12",0.65
48,2024-03,United Kingdom,Marketplace,756.16,17,15,"1,185.73",0.64
88,2024-05,United States,Online,"1,820.16",42,40,"2,861.03",0.64
34,2024-02,United States,Online,"2,388.37",55,52,"3,786.11",0.63
107,2024-06,United States,Store Pickup,845.22,20,20,"1,378.02",0.61
65,2024-04,Germany,Store Pickup,341.74,8,6,565.21,0.60


## Lab

Imagine the COO wants three tables by the end of the day:

### Table A — Country performance
Include:
- total sales
- transactions
- unique customers
- average sales
- return rate

### Table B — Product performance
Include:
- category
- supplier
- margin band
- total sales
- average sales
- return rate

### Table C — Monthly management dashboard
Rows:
- month

Columns:
- country

Values:
- total sales

Then convert Table C into long format for a future line chart.

Students should build each table from scratch.


In [58]:
# Student work area
# Build Table A, Table B, and Table C here



## Solution sketch


In [59]:
# Table A
table_a = (
    tx_all.groupby("country")
    .agg(
        total_sales=("net_sales", "sum"),
        transactions=("order_id", "count"),
        unique_customers=("customer_id", "nunique"),
        avg_sales=("net_sales", "mean"),
        return_rate=("is_return", "mean")
    )
    .sort_values("total_sales", ascending=False)
)
table_a

,total_sales,transactions,unique_customers,avg_sales,return_rate
country,,,,,
United States,"23,466.65",601,250,39.05,0.03
United Kingdom,"13,226.06",387,99,34.18,0.06
Germany,"10,761.01",355,54,30.31,0.05
Canada,"10,360.75",383,81,27.05,0.07
Australia,"9,449.97",358,54,26.40,0.06
France,"7,863.45",317,38,24.81,0.05


In [60]:
# Table B
table_b = (
    tx_all.groupby(["category", "supplier", "margin_band"])
    .agg(
        total_sales=("net_sales", "sum"),
        avg_sales=("net_sales", "mean"),
        return_rate=("is_return", "mean")
    )
    .reset_index()
    .sort_values("total_sales", ascending=False)
)
table_b.head(20)

,category,supplier,margin_band,total_sales,avg_sales,return_rate
8,Equipment,BlueRiver Imports,Medium,"8,970.54",52.46,0.07
6,Coffee,Summit Trading,High,"8,604.76",31.40,0.06
14,Tea,BlueRiver Imports,Low,"7,842.33",30.40,0.04
2,Bakery,NorthPeak Supply,High,"4,805.29",42.52,0.04
4,Coffee,BlueRiver Imports,Low,"4,620.93",30.20,0.05
10,Merch,Aurora Goods,Low,"4,520.28",33.48,0.06
1,Bakery,Maple Bridge,High,"4,147.43",36.06,0.01
12,Merch,BlueRiver Imports,Medium,"4,075.42",33.68,0.10
5,Coffee,NorthPeak Supply,Medium,"3,830.52",25.04,0.03
9,Equipment,Summit Trading,High,"3,761.04",37.24,0.06


In [61]:
# Table C
table_c = pd.pivot_table(
    tx_all,
    index="order_month",
    columns="country",
    values="net_sales",
    aggfunc="sum",
    fill_value=0
)
table_c

country,Australia,Canada,France,Germany,United Kingdom,United States
order_month,,,,,,
2024-01,"1,396.81","1,366.57","1,104.51","2,183.68","2,573.29","3,640.03"
2024-02,"1,655.06","1,742.50","1,192.81","1,615.85","2,390.16","4,221.71"
2024-03,"1,797.76","1,831.11","1,482.02","1,730.17","1,953.65","4,213.27"
2024-04,"1,437.84","1,857.13","1,196.34","1,832.66","2,354.35","4,393.60"
2024-05,"1,601.96","1,726.57","1,388.46","1,634.19","2,225.57","3,578.62"
2024-06,"1,560.54","1,836.87","1,499.31","1,764.46","1,729.04","3,419.42"


In [62]:
table_c_long = table_c.reset_index().melt(
    id_vars="order_month",
    var_name="country",
    value_name="total_sales"
)
table_c_long.head(15)

,order_month,country,total_sales
0,2024-01,Australia,"1,396.81"
1,2024-02,Australia,"1,655.06"
2,2024-03,Australia,"1,797.76"
3,2024-04,Australia,"1,437.84"
4,2024-05,Australia,"1,601.96"
5,2024-06,Australia,"1,560.54"
6,2024-01,Canada,"1,366.57"
7,2024-02,Canada,"1,742.50"
8,2024-03,Canada,"1,831.11"
9,2024-04,Canada,"1,857.13"


## Common mistakes to discuss

We often make these mistakes:

1. Using the wrong aggregation for the question  
2. Forgetting to reset index after groupby  
3. Merging on the wrong key  
4. Using an inner join when a left join is needed  
5. Forgetting to validate merge success  
6. Confusing wide and long formats  
7. Treating grouped output as if each row still represented a transaction

These are exactly the kinds of mistakes that appear in real analyst work.


## Reflection

1. When is `groupby()` more useful than a simple filter?
2. How do you choose between `count`, `sum`, `mean`, and `nunique`?
3. Why is merge-key selection so important?
4. When would you use a pivot table instead of a grouped summary?
5. Why might long format be better than wide format in later modules?


## Challenging Exercies

### Extension 1
Compute monthly sales growth by country.

### Extension 2
Find the top 3 products in each country by total sales.

### Extension 3
Create a table of customer lifetime value by acquisition channel.

### Extension 4
Estimate sales per campaign using the marketing table.

### Extension 5
Build a tidy summary table suitable for a future visualization module.


In [ ]:
# Extension 1
monthly_growth = pd.pivot_table(
    tx_all,
    index="order_month",
    columns="country",
    values="net_sales",
    aggfunc="sum",
    fill_value=0
).pct_change()
monthly_growth

In [ ]:
# Extension 2
top_products_country = (
    tx_all.groupby(["country", "product_name"])["net_sales"]
    .sum()
    .reset_index()
    .sort_values(["country", "net_sales"], ascending=[True, False])
)
top_products_country.groupby("country").head(3)

In [ ]:
# Extension 3
clv_channel = (
    tx_all.groupby(["customer_id", "acquisition_channel"])["net_sales"]
    .sum()
    .reset_index()
    .groupby("acquisition_channel")["net_sales"]
    .agg(["count", "mean", "median", "sum"])
    .sort_values("sum", ascending=False)
)
clv_channel

## Key takeaways

This session should leave students with a powerful practical workflow:

> **clean rows → summarize with groupby → enrich with merges → reshape for reporting**

That workflow is central to business analytics, dashboarding, and applied data science.

Students should now be able to:
- answer management questions with grouped summaries
- combine multiple sources correctly
- prepare tables for charts and presentations
- think more carefully about data grain and keys
